# CausaSent — Kaggle training notebook

Runs on a Kaggle GPU session (P100 16 GB or T4 ×2 30 GB, 12-hour limit, 30 GPU-hr/week).

**Setup before running:**
1. **Settings → Accelerator → GPU** (P100 or T4 ×2).
2. **Settings → Internet → On** (needed for `pip install`, HF Hub, and the VnCoreNLP download).
3. **Add-ons → Secrets → add `HF_TOKEN`** with *write* access — required to push the trained checkpoints back. Get a token at https://huggingface.co/settings/tokens.
4. (Optional) **Add `GEMINI_API_KEY`** if you want to run weak labeling inside the kernel.

**Pipeline:**
1. Clone the `feat/skeleton` (or `develop`) branch.
2. `pip install -r requirements.txt`.
3. Authenticate HF.
4. Pull the gold dataset from `Tamir39/causasent`.
5. Sanity-check env.
6. Train PhoBERT-large two-head tagger (best by entity-F1).
7. Train mT5-base action generator.
8. Push both checkpoints to `Tamir39/causasent-phobert` and `Tamir39/causasent-mt5`.

In [ ]:
# Cell 1 — Clone the repo into /kaggle/working
import os, subprocess

REPO_URL = 'https://github.com/tamir39/causa-sent.git'
REPO_DIR = '/kaggle/working/CausaSent'
BRANCH = 'feat/skeleton'  # switch to 'develop' once merged

if not os.path.isdir(REPO_DIR):
    subprocess.check_call(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, REPO_DIR])
else:
    subprocess.check_call(['git', '-C', REPO_DIR, 'pull', '--ff-only'])

os.chdir(REPO_DIR)
print(subprocess.check_output(['git', 'log', '-1', '--oneline']).decode().strip())

In [ ]:
# Cell 2 — Install dependencies (Kaggle base has torch/transformers; this upgrades and adds the rest).
%pip install -q -r requirements.txt 2>&1 | tail -10

In [ ]:
# Cell 3 — Authenticate with HF Hub via the Kaggle secret.
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
os.environ['HF_TOKEN'] = HF_TOKEN
login(token=HF_TOKEN, add_to_git_credential=False)
print('HF login OK')

In [ ]:
# Cell 4 — Path + env setup.
# Keep dataset *inside* the repo dir so configs/*.yaml relative paths just work.
import sys
ROOT = '/kaggle/working/CausaSent'
os.environ['CAUSASENT_DATA_DIR'] = f'{ROOT}/data'
os.environ['MPLBACKEND'] = 'Agg'
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

In [ ]:
# Cell 5 — GPU + library sanity check.
!python scripts/check_env.py

In [ ]:
# Cell 6 — Pull the gold dataset from HF Hub.
# Lays out: data/gold/{train,val,test}.json (and data/processed/weak.json if uploaded).
!python scripts/fetch_dataset.py --dest $CAUSASENT_DATA_DIR

In [ ]:
# Cell 7 — Train PhoBERT-large two-head tagger.
# Best checkpoint is saved by *entity-F1*, not val_loss (see src/train/train_phobert.py).
# Outputs: checkpoints/phobert/{best.pt,last.pt} and runs/<ts>-phobert/{config.yaml,git_sha.txt,env.txt,metrics.jsonl}.
!python -m src.train.train_phobert --config configs/phobert.yaml

In [ ]:
# Cell 8 — Train mT5-base action generator.
# Outputs: checkpoints/mt5/{best.pt,last.pt} and runs/<ts>-mt5/...
!python -m src.train.train_mt5 --config configs/mt5.yaml

In [ ]:
# Cell 9 — Eval both heads on the test split (entity-F1 + ROUGE-L).
!python -m src.eval.eval_phobert --config configs/phobert.yaml --ckpt checkpoints/phobert/best.pt --split test
print('---')
!python -m src.eval.eval_mt5 --config configs/mt5.yaml --ckpt checkpoints/mt5/best.pt --split test

In [ ]:
# Cell 10 — Push trained checkpoints + run metadata to HF Hub.
# Two separate model repos so they can be downloaded independently.
from pathlib import Path
from huggingface_hub import HfApi, create_repo

MODELS = {
    'Tamir39/causasent-phobert': 'checkpoints/phobert',
    'Tamir39/causasent-mt5': 'checkpoints/mt5',
}

api = HfApi()
for repo_id, local in MODELS.items():
    if not Path(local).is_dir():
        print(f'skip {repo_id}: no checkpoints at {local}')
        continue
    create_repo(repo_id, repo_type='model', exist_ok=True, private=False)
    api.upload_folder(
        folder_path=local,
        repo_id=repo_id,
        repo_type='model',
        commit_message=f'training run from Kaggle ({Path(local).name})',
        ignore_patterns=['**/__pycache__/**', '*.tmp'],
    )
    print(f'pushed -> https://huggingface.co/{repo_id}')

In [ ]:
# Cell 11 — Launch Gradio demo with a public share link.
# Uses the freshly trained checkpoints in /kaggle/working/CausaSent/checkpoints/.
# Click the gradio.live URL printed below to open the demo in a new tab.
# Note: the public link is valid for ~72h and tunnels through gradio's relay.
!python -m src.demo.app --share